In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [ ]:
os.makedirs("../../result/U2OS_HEK293T_permute/ranking/", exist_ok=True)

### Inputs: technical covariates of every real vector, the permutation, coverage class

In [ ]:
def load_input(infile):
    return pd.read_csv(infile, sep="\t", index_col=0)


def knn_mean_cosine(unit, k, chunk=2000):
    """
    Mean cosine of every row to its k nearest rows, self excluded.
    """
    unit = unit.astype(np.float32)
    n = unit.shape[0]
    out = np.empty(n, dtype=np.float64)
    for start in range(0, n, chunk):
        stop = min(start + chunk, n)
        sim = unit[start:stop] @ unit.T
        sim[np.arange(stop - start), np.arange(start, stop)] = -np.inf
        top = np.partition(sim, n - k, axis=1)[:, n - k:]
        out[start:stop] = top.mean(axis=1)
    return out


def file_covariates(emb):
    """
    One row per protein of one input embedding: norm, hubness.
    """
    values = emb.values.astype(np.float64)
    norm = np.linalg.norm(values, axis=1)
    unit = values / np.maximum(norm, 1e-12)[:, None]
    centroid = values.mean(axis=0)
    centroid = centroid / max(np.linalg.norm(centroid), 1e-12)
    frame = pd.DataFrame(index=emb.index)
    frame["norm"] = norm
    frame["cos_centroid"] = unit @ centroid
    frame["knn_cos"] = knn_mean_cosine(unit, 10)
    return frame


def recover_permutation(real, perm, tol=1e-9, chunk=2000):
    """
    if permuted name A carries the original vector of B, it records {A:B}
    """
    R = real.values.astype(np.float64)
    P = perm.values.astype(np.float64)
    r_sq = (R ** 2).sum(axis=1)
    best = np.empty(len(P), dtype=np.int64)
    for start in range(0, len(P), chunk):
        stop = min(start + chunk, len(P))
        dist = r_sq[None, :] - 2 * P[start:stop] @ R.T
        best[start:stop] = dist.argmin(axis=1)
    gap = np.abs(P - R[best]).max(axis=1)
    claims = np.bincount(best, minlength=len(R))
    ok = (gap <= tol) & (claims[best] == 1)
    return pd.Series(real.index[best[ok]], index=perm.index[ok])


covariates = {}
permute_map = {}
real_index = {}
for file, infile_real, infile_perm in [
    (
        "APMS-U2OS",
        "../../result/U2OS_HEK293T/input/APMS-U2OS.tsv",
        "../../result/U2OS_HEK293T_permute/input/APMS-U2OS.tsv"
    ),
    (
        "Image-U2OS",
        "../../result/U2OS_HEK293T/input/Image-U2OS.tsv",
        "../../result/U2OS_HEK293T_permute/input/Image-U2OS.tsv"
    ),
    (
        "APMS-HEK293T",
        "../../result/U2OS_HEK293T/input/APMS-HEK293T.tsv",
        "../../result/U2OS_HEK293T_permute/input/APMS-HEK293T.tsv"
    ),
    (
        "Image-HEK293T",
        "../../result/U2OS_HEK293T/input/Image-HEK293T.tsv",
        "../../result/U2OS_HEK293T_permute/input/Image-HEK293T.tsv"
    ),
]:
    real = load_input(infile_real)
    perm = load_input(infile_perm)
    assert set(real.index) == set(perm.index), file
    real_index[file] = real.index
    covariates[file] = file_covariates(real)
    permute_map[file] = recover_permutation(real, perm)
    
identity = {
    file: pd.Series(cov.index, index=cov.index)
    for file, cov in covariates.items()
}

In [ ]:
# coverage class of every query protein with a base counterpart
query_names = pd.Index(sorted(
    set(real_index["APMS-U2OS"])
    | set(real_index["Image-U2OS"])
))
coverage = pd.DataFrame({
    "APMS-U2OS": query_names.isin(real_index["APMS-U2OS"]),
    "Image-U2OS": query_names.isin(real_index["Image-U2OS"]),
    "APMS-HEK293T": query_names.isin(real_index["APMS-HEK293T"]),
    "Image-HEK293T": query_names.isin(real_index["Image-HEK293T"]),
}, index=query_names).astype(int)
cls_all = coverage.astype(str).agg("".join, axis=1)
names = cls_all.index[coverage[
    ["APMS-HEK293T", "Image-HEK293T"]
].any(axis=1)]
cls = cls_all.reindex(names)

### Scores of the real and the permuted run

In [ ]:
def load_score(infile):
    if infile.endswith(".cca"):
        frame = pd.read_csv(infile, sep="\t")
        return frame.set_index("gene")["score"].astype(float)
    frame = pd.read_csv(infile, sep="\t", index_col=0)
    return frame["specific_score"].astype(float)

### Technical model on the permuted run, removed from the real score within coverage class

In [ ]:
def design_matrix(index, files, carrier):
    """
    construct feature.
    carrier[file] maps a name to the real protein whose vector it carries:
    the identity for the real run, the recovered permutation for the permuted run.
    """
    blocks = []
    for file, slot in files:
        carried = carrier[file].reindex(index)
        block = covariates[file].reindex(carried.values)[
            ["norm", "cos_centroid", "knn_cos"]
        ]
        block.index = index
        block.columns = [f"{slot}_{f}" for f in block.columns]
        blocks.append(block)
    return pd.concat(blocks, axis=1)


def technical_expectation(perm_score):
    """
    OLS of the permuted score on the covariates
    """
    tech = pd.Series(np.nan, index=names, dtype=float)
    r2 = {}
    perm_cls = cls_all.reindex(perm_score.index).values
    for c, files in {
        c: [
            (file, slot)
            for (file, slot), bit in zip([
                ("APMS-U2OS", "qA"),
                ("Image-U2OS", "qI"),
                ("APMS-HEK293T", "bA"),
                ("Image-HEK293T", "bI")
            ], c)
            if bit == "1"
        ]
        for c in sorted(cls.unique())
    }.items():
        perm_names = perm_score.index[perm_cls == c]
        X_perm = design_matrix(perm_names, files, permute_map)
        y = perm_score.reindex(perm_names)
        keep = X_perm.notna().all(axis=1) & y.notna()
        if keep.sum() < 10 * X_perm.shape[1]:
            continue
        model = LinearRegression().fit(
            X_perm[keep].values,
            y[keep].values,
        )
        fitted = model.predict(X_perm[keep].values)
        ss_res = ((y[keep] - fitted) ** 2).sum()
        ss_tot = ((y[keep] - y[keep].mean()) ** 2).sum()
        r2[c] = 1 - ss_res / ss_tot

        real_names = names[cls.values == c]
        X_real = design_matrix(real_names, files, identity)
        ok = X_real.notna().all(axis=1)
        pred = model.predict(X_real[ok].values)
        tech.loc[real_names[ok.values]] = pred
    return tech, r2


def remove_technical(real, tech):
    """
    The permute-removed score:
    real score minus its OLS fit on the technical expectation.
    """
    out = pd.Series(np.nan, index=real.index)
    for c in cls.unique():
        in_cls = (cls == c) & real.notna()
        mask = in_cls & tech.notna()
        if mask.sum() < 20 or tech[mask].std() < 1e-12:
            out[in_cls] = real[in_cls] - real[in_cls].mean()
            continue
        slope, intercept = np.polyfit(
            tech[mask].values,
            real[mask].values,
            1,
        )
        out[mask] = real[mask] - (intercept + slope * tech[mask])
    return out

### Output: one ranking per group

In [ ]:
for infile_real, infile_perm, outfile in [
    (
        "../../result/U2OS_HEK293T/cca/FromInputConcat_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/cca/FromInputConcat_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/ranking/FromInputConcat_U2OS.ranking"
    ),
    (
        "../../result/U2OS_HEK293T/cca/AEconcat_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/cca/AEconcat_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/ranking/AEconcat_U2OS.ranking"
    ),
    (
        "../../result/U2OS_HEK293T/cca/GPSseparate_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/cca/GPSseparate_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/ranking/GPSseparate_U2OS.ranking"
    ),
    (
        "../../result/U2OS_HEK293T/cca/GPScombine_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/cca/GPScombine_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/ranking/GPScombine_U2OS.ranking"
    ),
    (
        "../../result/U2OS_HEK293T/cca/promergesubtract_HEK293Tanchor_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/cca/promergesubtract_HEK293Tanchor_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/ranking/promergesubtract_HEK293Tanchor_U2OS.ranking"
    ),
    (
        "../../result/U2OS_HEK293T/cca/promergeclub_HEK293Tanchor_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/cca/promergeclub_HEK293Tanchor_U2OS.cca",
        "../../result/U2OS_HEK293T_permute/ranking/promergeclub_HEK293Tanchor_U2OS.ranking"
    ),
    (
        "../../result/U2OS_HEK293T/coemb/promergeclub_HEK293Tanchor/U2OS/U2OS_specific_score.tsv",
        "../../result/U2OS_HEK293T_permute/coemb/promergeclub_HEK293Tanchor/U2OS/U2OS_specific_score.tsv",
        "../../result/U2OS_HEK293T_permute/ranking/promergeclub_HEK293Tanchor_U2OS_specific_score.ranking"
    ),
]:
    real = load_score(infile_real).reindex(names)
    perm = load_score(infile_perm)
    tech, r2 = technical_expectation(perm)
    score = remove_technical(real, tech)

    out = pd.DataFrame({
        "gene": names,
        "score": score.values,
        "rank": score.rank(ascending=False, method="min").values,
        "rank_in_class": score.groupby(cls).rank(
            ascending=False, method="min"
        ).values,
        "coverage_class": cls.values,
        "real_score": real.values,
        "technical_expectation": tech.values,
    }).dropna(subset=["score"]).sort_values("rank")
    out["rank"] = out["rank"].astype(int)
    out["rank_in_class"] = out["rank_in_class"].astype(int)
    out.to_csv(outfile, sep="\t", index=False)